In [16]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import optuna
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import log_loss
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import StratifiedKFold

In [17]:
#Load Data
train = pd.read_csv('EV_train.csv')
test = pd.read_csv('EV_test.csv')
sample_submission = pd.read_csv('EV_sample_submission.csv')

target = "Will_Buy_EV"
train[target] = train[target].map({'No' : 0,
                                    'Yes' : 1
})

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("Sample Submission Shape:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

Train Shape: (668665, 15)
Test Shape: (286571, 14)
Sample Submission Shape: (286571, 2)


,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,0
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,0
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,1
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,0
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,0


,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low


,id,Will_Buy_EV
0,668665,0.174645
1,668666,0.174645
2,668667,0.174645
3,668668,0.174645
4,668669,0.174645


In [18]:
features = [column for column in test.columns if column != 'id']
print("Number Of Featrues:", len(features))

numerical_features = train[features].select_dtypes(include='number').columns.tolist()
print('Numerical Features:')
print(numerical_features)

print()

categorical_features = train[features].select_dtypes(exclude='number').columns.tolist()
print('Categorical Features:')
print(categorical_features)

Number Of Featrues: 13
Numerical Features:
['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']

Categorical Features:
['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [19]:
Xtrain = train[features]
Xtest = test[features]
ytrain = train[target]

print(f"X train: {Xtrain.shape} X test: {Xtest.shape} ytrain: {ytrain.shape}")

X train: (668665, 13) X test: (286571, 13) ytrain: (668665,)


In [20]:
#Pre processing
for col in categorical_features:
    Xtrain[col] = Xtrain[col].astype("category")
    Xtest[col] = Xtest[col].astype("category")
    # Align category sets between train/test so encodings match
    cats = pd.api.types.union_categoricals(
        [Xtrain[col], Xtest[col]]
    ).categories
    Xtrain[col] = Xtrain[col].cat.set_categories(cats)
    Xtest[col] = Xtest[col].cat.set_categories(cats)

In [ ]:
def objective(trial):
    params = {
        "objective": "binary:logistic",   # binary classification
        "eval_metric": "auc",
        "booster": "gbtree",
        "tree_method": "hist",
        "enable_categorical": True,
        "verbosity": 0,
        
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for fold_idx, (train_idx, valid_idx) in enumerate(cv.split(Xtrain, ytrain)):
        X_tr, X_val = Xtrain.iloc[train_idx], Xtrain.iloc[valid_idx]
        y_tr, y_val = ytrain.iloc[train_idx], ytrain.iloc[valid_idx]

        model = xgb.XGBClassifier(**params, random_state=42, n_jobs=-1)
        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, pred)
        scores.append(auc)

        # report running mean so far, keyed by fold index
        trial.report(np.mean(scores), step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)

    study = optuna.create_study(
        direction="maximize",  # maximize ROC AUC (higher is better)
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
    )
    study.optimize(objective, n_trials=5, timeout=3600)

    print("Best params:", study.best_params)
    print("Best CV ROC AUC:", study.best_value)

[I 2026-09-06 18:19:38,377] A new study created in memory with name: no-name-9e43c146-bf19-46e4-a738-8cb464a38cef
[I 2026-09-06 18:20:48,213] Trial 0 finished with value: 0.940446891479563 and parameters: {'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.01316964467230591, 'subsample': 0.8065372421127094, 'colsample_bytree': 0.9496964938696428, 'min_child_weight': 6, 'gamma': 4.196316626994113, 'reg_alpha': 8.162428927680336, 'reg_lambda': 1.3427121527270705e-08}. Best is trial 0 with value: 0.940446891479563.
[I 2026-09-06 18:21:46,606] Trial 1 finished with value: 0.9418859129061847 and parameters: {'n_estimators': 650, 'max_depth': 5, 'learning_rate': 0.06257167724513675, 'subsample': 0.8386871607630706, 'colsample_bytree': 0.7482019339639725, 'min_child_weight': 9, 'gamma': 1.8400273586065397, 'reg_alpha': 4.315723283410588, 'reg_lambda': 7.911824539411945}. Best is trial 1 with value: 0.9418859129061847.
[I 2026-09-06 18:22:21,210] Trial 2 finished with value: 0.94155112527

Best params: {'n_estimators': 650, 'max_depth': 5, 'learning_rate': 0.06257167724513675, 'subsample': 0.8386871607630706, 'colsample_bytree': 0.7482019339639725, 'min_child_weight': 9, 'gamma': 1.8400273586065397, 'reg_alpha': 4.315723283410588, 'reg_lambda': 7.911824539411945}
Best CV ROC AUC: 0.9418859129061847


In [29]:
best_model = xgb.XGBClassifier(**study.best_params)
best_model.fit(Xtrain, ytrain)

predictions = best_model.predict_proba(Xtest)[:, 1]

In [30]:
submission = sample_submission.copy()
submission[target] = predictions
submission[target].describe().to_frame()

,Will_Buy_EV
count,286571.000000
mean,0.174674
std,0.269490
min,0.000035
25%,0.002793
50%,0.018001
75%,0.267827
max,0.970237


In [31]:
submission.to_csv('submission.csv', index=False)
print("It worked")

It worked
